### Мы имеем таблицу с продажами пленок (одна строка - продажа одной пленки). Переделаем эту таблицу в формат (одна строка - дата, станция и количество пленок, которое продала эта станция в эту дату)

In [47]:
from tarfile import data_filter

import pandas as pd


sales_initial = pd.read_csv('../../data/initial_data/ps_stats_initial.csv', parse_dates=['created', 'cut_time'])

In [48]:
sales_initial

,id,created,cut_time,film_size,film_type,sale_type,time_zone,element_id,station_id
0,1426826,2026-09-16 08:33:01.203202,2026-09-16 11:32:58.067,PHONE,GLOSS,SALE,GMT+3:00,18828.0,2417.0
1,1426825,2026-09-16 08:32:53.050035,2026-09-16 11:32:50.449,PHONE,MATE,SALE,GMT+3:00,15795.0,2854.0
2,1426824,2026-09-16 08:31:40.230850,2026-09-16 13:31:40.030,PHONE,MATE,SALE,GMT+5:00,18268.0,438.0
3,1426823,2026-09-16 08:30:57.475055,2026-09-16 15:30:56.323,PHONE,MATE,DEFECT,GMT+7:00,16009.0,2265.0
4,1426822,2026-09-16 08:30:42.547585,2026-09-16 11:30:41.074,WATCH,GLOSS,SALE,GMT+3:00,17191.0,2224.0
...,...,...,...,...,...,...,...,...,...
1186807,240028,2023-07-27 13:15:28.000000,2023-07-27 07:15:25.918,PHONE,GLOSS,DEFECT,GMT 3:00,NaN,1.0
1186808,240027,2023-07-27 13:11:28.000000,2023-07-27 07:11:25.334,PHONE,GLOSS,DEFECT,GMT 5:00,NaN,586.0
1186809,240026,2023-07-26 23:50:06.000000,2023-07-26 15:22:41.211,LAPTOP,GLOSS,SALE,GMT 3:00,NaN,1.0
1186810,240025,2023-07-26 23:49:04.000000,2023-07-26 15:22:41.222,LAPTOP,GLOSS,SALE,GMT 3:00,NaN,1.0


In [49]:
sales_initial.info()

<class 'pandas.DataFrame'>
RangeIndex: 1186812 entries, 0 to 1186811
Data columns (total 9 columns):
 #   Column      Non-Null Count    Dtype         
---  ------      --------------    -----         
 0   id          1186812 non-null  int64         
 1   created     1186812 non-null  datetime64[us]
 2   cut_time    1186812 non-null  datetime64[us]
 3   film_size   1186812 non-null  str           
 4   film_type   1186812 non-null  str           
 5   sale_type   1186812 non-null  str           
 6   time_zone   1186812 non-null  str           
 7   element_id  919061 non-null   float64       
 8   station_id  1186757 non-null  float64       
dtypes: datetime64[us](2), float64(2), int64(1), str(4)
memory usage: 81.5 MB


In [50]:
sales = sales_initial.copy()
sales = sales.drop(['id', 'created', 'film_size', 'film_type', 'time_zone', 'element_id'], axis=1)
sales = sales[sales['sale_type'] == 'SALE']
sales['date'] = sales['cut_time'].dt.date
sales = sales.drop(['cut_time', 'sale_type'], axis=1)
sales['daily_sales_count'] = sales.groupby(['station_id', 'date'])['date'].transform('count')
sales = sales.drop_duplicates().sort_values(by=['date', 'station_id']).reset_index(drop=True)

sales

,station_id,date,daily_sales_count
0,2581.0,2021-12-19,1.0
1,2540.0,2022-01-10,1.0
2,2878.0,2022-01-10,5.0
3,396.0,2022-02-14,1.0
4,2263.0,2022-02-14,1.0
...,...,...,...
515885,3918.0,2026-09-16,1.0
515886,3930.0,2026-09-16,1.0
515887,3974.0,2026-09-16,1.0
515888,4001.0,2026-09-16,2.0


### Известно, что некоторые станции носят технический и тестовый характер, поэтому их стоит исключить

In [51]:
stations_to_delete = {
    495, 1283, 1288, 1800, 1802, 1803, 1822, 1880, 1881, 1934,
    1936, 1947, 1950, 1951, 1952, 1954, 1971, 2224, 2242, 2243,
    2247, 2342, 2343, 2345, 2392, 2433, 2450, 2562, 2593, 2760,
    2902, 2903, 2904, 2910, 3267, 3268, 3270, 3300, 3353, 3354,
    3366, 3441, 3488, 3494, 3511, 3554, 3577, 3585, 3619, 3630,
    3636, 3637, 3638, 1227, 1538, 1539, 1540, 1805, 1824, 1826,
    1828, 1926, 1957, 1972, 2254, 3316, 3361, 3508, 3602, 1801, 1
}

sales = sales[~sales['station_id'].isin(stations_to_delete)]

### Посмотрим на распределение того, сколько в день продают пленок (нам станет ясно, что есть выбросы, которые стоит исключить)

In [52]:
sales['daily_sales_count'].value_counts()

daily_sales_count
1.0      284754
2.0      123202
3.0       52909
4.0       23237
5.0       11014
6.0        5417
7.0        2904
8.0        1505
9.0         930
10.0        554
11.0        362
12.0        216
13.0        181
14.0         94
15.0         72
16.0         67
17.0         34
19.0         27
18.0         23
20.0         20
22.0         17
21.0         16
24.0          7
27.0          7
23.0          7
32.0          6
29.0          6
26.0          5
25.0          4
37.0          4
42.0          3
36.0          3
30.0          3
40.0          2
31.0          2
28.0          2
35.0          2
280.0         1
467.0         1
50.0          1
62.0          1
38.0          1
33.0          1
44.0          1
39.0          1
54.0          1
Name: count, dtype: int64

In [53]:
sales = sales[sales['daily_sales_count'] <= 20]

### Теперь остается разбить данные на train and test, в качестве тестовой выборки будем использовать август 2026 года.

In [54]:
sales

,station_id,date,daily_sales_count
0,2581.0,2021-12-19,1.0
1,2540.0,2022-01-10,1.0
2,2878.0,2022-01-10,5.0
3,396.0,2022-02-14,1.0
4,2263.0,2022-02-14,1.0
...,...,...,...
515885,3918.0,2026-09-16,1.0
515886,3930.0,2026-09-16,1.0
515887,3974.0,2026-09-16,1.0
515888,4001.0,2026-09-16,2.0


In [55]:
train_sales = sales[sales['date'] <= pd.to_datetime('2026-07-31').date()]
test_sales = sales[sales['date'] > pd.to_datetime('2026-07-31').date()]